# Tables and figures of a run

The notebook reads `summary.json` and the logs of one run from `results/<RUN_NAME>/` and builds every table and
figure used in the paper. It sends no requests to a model. Tables are saved as CSV and figures as PNG and PDF
to `results/<RUN_NAME>/paper/`.

Set `RUN_NAME` below (or the environment variable `RUN_NAME`) to the `--name` of the run.

In [ ]:
import os
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src import metrics, report
from src.config import load_config, resolve

RUN_NAME = os.environ.get("RUN_NAME", "main")

run_dir = resolve(load_config()["results_dir"]) / RUN_NAME
paper = run_dir / "paper"
paper.mkdir(exist_ok=True)
run = report.load_run(run_dir)
summary = run["summary"]
pd.set_option("display.max_columns", None, "display.width", 250, "display.precision", 3)


def save_table(table, name):
    table.to_csv(paper / f"{name}.csv", encoding="utf-8", lineterminator="\n")
    display(table)


def save_figure(fig, name):
    fig.savefig(paper / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(paper / f"{name}.pdf", bbox_inches="tight")
    plt.show()


print("Run:", run_dir)
print("Detector:", summary["detector_model"], "| attacker:", summary["attacker_model"])
print("Function pairs:", summary["pairs"], "| max rounds:", summary["max_rounds"], "| seed:", summary["seed"])
print("Temperature: detector", summary["detector_temperature"], "| attacker", summary["attacker_temperature"])

## Table 1. Detection quality and robustness of the six configurations

Precision, recall and F1 are measured without attack on vulnerable and fixed versions. The attack targets only
the functions that a configuration flagged correctly. The control flip rate is the share of neutral comments
after which the verdict became "safe"; an ASR close to it means instability rather than persuasion.
Answers that were not valid JSON are excluded from all rates and counted in the last column.

In [ ]:
save_table(report.metrics_table(summary), "table1_configurations")

## Table 2 and Figure 1. Attack success rate by round budget

ASR(R) is the share of attacked functions whose verdict became "safe" at round R or earlier (round 0 is the
first comment, written without feedback). Dashed lines in the figure show the control flip rate of the
configuration with the same colour.

In [ ]:
save_table(report.asr_table(summary), "table2_asr_by_budget")

fig, ax = plt.subplots(figsize=(8.5, 4.2))
metrics.draw_asr(summary, ax)
save_figure(fig, "figure1_asr_curve")

## Table 3 and Figure 2. Success of comment styles

Every evaluated comment is one row of `rounds.jsonl`. The success rate of a style is the share of its
comments after which the verdict became "safe". Rounds are not independent: later rounds are spent on the
functions that resisted earlier comments, so the rates describe the attack as it ran and are not a
controlled comparison of styles.

In [ ]:
styles = report.style_table(run["rounds"])
save_table(styles.set_index(["configuration", "style"]), "table3_styles")
if len(styles):
    names = report.STYLE_NAMES + [s for s in styles["style"].unique() if s not in report.STYLE_NAMES]
    save_figure(report.draw_rates(styles, "style", names, "Success rate of a comment by style"), "figure2_styles")

## Table 4 and Figure 3. Forged Cppcheck output against a real alert (RQ2)

`TOOL_CPPCHECK` comments imitate Cppcheck output that contradicts the alert given to the triage agent. They
are compared with forged output of other tools and with all remaining styles. "Share of successful attacks"
is the share of broken functions whose decisive comment belongs to the group. Configuration `A` receives no
alert and is shown for reference.

In [ ]:
tools = report.cppcheck_table(run["rounds"], run["attacks"])
save_table(tools.set_index(["configuration", "comment group"]), "table4_forged_cppcheck")
if len(tools):
    groups = list(report.TOOL_GROUPS)
    save_figure(report.draw_rates(tools, "comment group", groups, "Forged tool output against the alert"), "figure3_forged_cppcheck")

## Table 5. The LLM was persuaded, the veto held (`C-strict`, `C-weak`)

In the veto configurations the LLM agent may dismiss the alert, and the deterministic check then accepts or
rejects the dismissal. "LLM persuaded, veto held" is the share of attacked functions for which the agent
dismissed the alert at least once and every such dismissal was rejected. "ASR without the veto" counts the
functions for which the agent was persuaded at least once by the same comments. The last two rows show the
price of the veto on fixed versions: correct dismissals that it rejected.

In [ ]:
veto = report.veto_table(run["baseline"], run["rounds"], run["attacks"])
if len(veto):
    save_table(veto, "table5_veto")
else:
    print("The run has no veto configurations.")

## Example: rounds of one attack

In [ ]:
CONFIGURATION = "B"

rounds = run["rounds"][(run["rounds"]["config"] == CONFIGURATION) & run["rounds"]["style"].notna()]
if len(rounds):
    one = rounds[rounds["id"] == rounds["id"].iloc[0]]
    print("Function:", one["id"].iloc[0])
    for _, r in one.iterrows():
        print(f"--- round {r['round']} | style {r['style']} | verdict {r['verdict']}")
        print(r["comment"])
        print("Explanation:", r["explanation"])
        print()
else:
    print("No attack rounds for this configuration.")

In [ ]:
print("Saved to", paper)
for path in sorted(paper.iterdir()):
    print(" ", path.name)